# NepConformer Fine-tuning on Semi-Supervised Newari Data (14.72 hours)

**Research:** *Nwāchā Munā — Proximal Cross-Lingual Transfer for Ultra-Low-Resource Nepal Bhasha ASR*

This notebook trains NepConformer using a semi-supervised learning framework. It combines the foundational 5.39 hours of manually transcribed data with 9.33 hours of pseudo-labeled data. The model architecture, optimizer, and hyperparameters remain identical to `01_finetune_base.ipynb` to provide a clean ablation on the impact of incorporating uncurated, pseudo-labeled conversational speech.

---

## Relationship to Other Experiments

| Notebook | Training data | CER | WER |
|---|---|---|---|
| `01_finetune_base.ipynb` | 5.39 hrs (original) |18.72%  
| **`05_semisupervised_finetune.ipynb`** | **5.39 hrs labelled + 9.33 hrs pseudo-labeled** |**19.83%** .

*Note: Performance degraded in this specific experiment. As analyzed in the paper, this decline highlights the challenge of severe domain shift introduced by uncurated data (extensive code-mixing, overlapping speech, and highly variable acoustic environments) which overshadowed the benefits of increased training volume.*


## Notebook Overview

| Step | Description |
|------|-------------|
| 1 | Install dependencies |
| 2 | Configure experiment tracking (WandB) |
| 3 | Clone NVIDIA NeMo repository |
| 4 | Merge labelled and pseudo-labeled JSON manifests |
| 5 | Define training configuration |
| 6 | Patch NeMo training script |
| 7 | Launch fine-tuning |

**Environment:** NVIDIA Tesla T4 · NeMo 2.1.0 · Kaggle GPU notebook

---
## Step 1 — Install Dependencies

In [ ]:
!pip install wandb
!pip install pybind11
!pip install fasttext
!pip install "nemo_toolkit[asr]==2.1.0"

---
## Step 2 — Configure Experiment Tracking (Weights & Biases)

In [ ]:
import os

# Replace with your own key from https://wandb.ai/authorize
WANDB_API_KEY = "YOUR_WANDB_API_KEY"

with open("api.txt", "w") as f:
    f.write(WANDB_API_KEY)

with open("api.txt") as f:
    os.environ["WANDB_API_KEY"] = f.read().strip()

print("WandB API key loaded.")

---
## Step 3 — Clone NVIDIA NeMo Repository

In [ ]:
!git clone https://github.com/NVIDIA/NeMo.git

---
## Step 4 — Convert Val/Test Audio to Mono



> If you ran `01_finetune_base.ipynb` earlier in this Kaggle session, the mono val/test manifests already exist in `/kaggle/working/` and this step can be skipped.

In [ ]:
import json
import os
import soundfile as sf
import numpy as np
from tqdm import tqdm

INPUT_DIR   = "/kaggle/input/final-data"
MONO_DIR    = "/kaggle/working/mono_audio"
WORKING_DIR = "/kaggle/working"

os.makedirs(MONO_DIR, exist_ok=True)

# Only val and test — train comes from the augmented Kaggle dataset
EVAL_MANIFESTS = ["nemo_val.json", "nemo_test.json"]


def convert_to_mono(manifest_filename):
    """Read a NeMo manifest, convert referenced audio to mono, write an updated manifest."""
    input_path  = os.path.join(INPUT_DIR, manifest_filename)
    output_path = os.path.join(WORKING_DIR, f"mono_{manifest_filename}")

    if not os.path.exists(input_path):
        print(f"[SKIP] {manifest_filename} not found.")
        return

    updated_entries = []
    print(f"[Processing] {manifest_filename}")

    with open(input_path, "r", encoding="utf-8") as f:
        for line in tqdm(f, desc="  Converting"):
            entry = json.loads(line)
            orig  = entry["audio_filepath"]
            full  = orig if os.path.exists(orig) else os.path.join(INPUT_DIR, orig)
            dest  = os.path.join(MONO_DIR, os.path.basename(full))

            try:
                audio, sr = sf.read(full)
                if audio.ndim > 1 and audio.shape[1] > 1:
                    audio = audio.mean(axis=1)
                sf.write(dest, audio, sr)
                entry["audio_filepath"] = dest
                updated_entries.append(entry)
            except Exception as e:
                print(f"  [ERROR] {os.path.basename(full)}: {e}")

    with open(output_path, "w", encoding="utf-8") as f_out:
        for entry in updated_entries:
            f_out.write(json.dumps(entry, ensure_ascii=False) + "\n")

    print(f"  Done → {output_path}")


for manifest in EVAL_MANIFESTS:
    convert_to_mono(manifest)

print("\nVal and test sets ready.")

---
## Step 5 — Define Training Configuration



In [ ]:
config = """
name: "Conformer-CTC-BPE-Finetune-Newari-Augmented"

# Checkpoint init is handled via CLI: ++init_from_ptl_ckpt=<path>
init_from_ptl_ckpt: null

model:
  sample_rate: 16000
  log_prediction: true
  ctc_reduction: 'mean_batch'
  skip_nan_grad: false
  use_cer: true

  # ── Data ────────────────────────────────────────────────────────────────────
  train_ds:
    # Augmented manifest produced by 03_augmentation.ipynb, uploaded as Kaggle dataset
    manifest_filepath: "/kaggle/input/augmented-data/augmentation/nemo_train.json"
    sample_rate: ${model.sample_rate}
    batch_size: 4
    shuffle: true
    num_workers: 4
    pin_memory: true
    max_duration: 17.5      # raised from 16.7 to cover 0.9x slower copies (~11% longer)
    min_duration: 0.1
    is_tarred: false
    tarred_audio_filepaths: null
    shuffle_n: 2048
    bucketing_strategy: "synced_randomized"
    bucketing_batch_size: null

  validation_ds:
    manifest_filepath: "/kaggle/working/mono_nemo_val.json"   # original, never augmented
    sample_rate: ${model.sample_rate}
    batch_size: 4
    shuffle: false
    use_start_end_token: false
    num_workers: 4
    pin_memory: true

  test_ds:
    manifest_filepath: "/kaggle/working/mono_nemo_test.json"  # original, never augmented
    sample_rate: ${model.sample_rate}
    batch_size: 4
    shuffle: false
    use_start_end_token: false
    num_workers: 4
    pin_memory: true

  # ── Tokenizer (same pre-trained Newari BPE vocabulary) ──────────────────────
  tokenizer:
    dir: "/kaggle/input/datasets/jennypoudel/newari-data/"
    type: bpe

  # ── Feature Extraction ──────────────────────────────────────────────────────
  preprocessor:
    _target_: nemo.collections.asr.modules.AudioToMelSpectrogramPreprocessor
    sample_rate: ${model.sample_rate}
    normalize: "per_feature"
    window_size: 0.025
    window_stride: 0.01
    window: "hann"
    features: 80
    n_fft: 512
    log: true
    frame_splicing: 1
    dither: 0.00001
    pad_to: 0
    pad_value: 0.0

  # ── SpecAugment applied on top of offline augmentation ──────────────────────
  # Offline augmentation = waveform-level diversity
  # SpecAugment          = feature-level regularization
  spec_augment:
    _target_: nemo.collections.asr.modules.SpectrogramAugmentation
    freq_masks: 2
    time_masks: 2
    freq_width: 27
    time_width: 70

  # ── Encoder (18-layer Conformer, identical to base experiment) ───────────────
  encoder:
    _target_: nemo.collections.asr.modules.ConformerEncoder
    feat_in: ${model.preprocessor.features}
    feat_out: -1
    n_layers: 18
    d_model: 256
    subsampling: striding
    subsampling_factor: 4
    subsampling_conv_channels: -1
    causal_downsampling: false
    ff_expansion_factor: 4
    self_attention_model: rel_pos
    n_heads: 4
    att_context_size: [-1, -1]
    att_context_style: regular
    xscaling: true
    untie_biases: true
    pos_emb_max_len: 5000
    conv_kernel_size: 31
    conv_norm_type: 'batch_norm'
    conv_context_size: null
    dropout: 0.1
    dropout_pre_encoder: 0.1
    dropout_emb: 0.0
    dropout_att: 0.1
    stochastic_depth_drop_prob: 0.0
    stochastic_depth_mode: linear
    stochastic_depth_start_layer: 1

  decoder:
    _target_: nemo.collections.asr.modules.ConvASRDecoder
    feat_in: null
    num_classes: -1
    vocabulary: []

  interctc:
    loss_weights: []
    apply_at_layers: []

  # ── Optimizer (identical to base experiment) ─────────────────────────────────
  optim:
    name: adamw
    lr: 0.0001
    betas: [0.9, 0.98]
    weight_decay: 1e-3
    sched:
      name: CosineAnnealing
      min_lr: 1e-6
      warmup_steps: 3000

# ── Trainer ──────────────────────────────────────────────────────────────────
trainer:
  devices: -1
  num_nodes: 1
  max_epochs: 100
  max_steps: -1
  val_check_interval: 1.0
  accelerator: auto
  accumulate_grad_batches: 8
  gradient_clip_val: 1.0
  precision: 32
  log_every_n_steps: 100
  enable_progress_bar: True
  num_sanity_val_steps: 0
  check_val_every_n_epoch: 1
  sync_batchnorm: true
  enable_checkpointing: False
  logger: false
  benchmark: false
  max_time: "00:11:30:00"

# ── Experiment Manager ────────────────────────────────────────────────────────
exp_manager:
  exp_dir: null
  name: ${name}
  create_tensorboard_logger: true
  create_checkpoint_callback: true
  checkpoint_callback_params:
    monitor: "val_wer"
    mode: "min"
    save_top_k: 3
    always_save_nemo: True
  create_early_stopping_callback: False
  resume_if_exists: false
  resume_ignore_no_checkpoint: false
  create_wandb_logger: True
  wandb_logger_kwargs:
    name: "finetune-newari-augmented-23hrs"
    project: "Conformer-Finetuning"
"""

config_path = "finetune_aug_config.yaml"
with open(config_path, "w") as f:
    f.write(config)

print(f"Training config written to: {config_path}")

---
## Step 6 — Patch NeMo Training Script



In [ ]:
NEMO_SCRIPT_PATH = "NeMo/examples/asr/asr_ctc/speech_to_text_ctc_bpe.py"

patched_script = '''
# Copyright (c) 2020, NVIDIA CORPORATION.  All rights reserved.
# Licensed under the Apache License, Version 2.0.

import lightning.pytorch as pl
from omegaconf import OmegaConf
import torch
import numpy as np

# ── Compatibility Fix 1: NumPy 2.0 removed np.sctypes ────────────────────────
if not hasattr(np, "sctypes"):
    np.sctypes = {
        "int":     [np.int8, np.int16, np.int32, np.int64],
        "uint":    [np.uint8, np.uint16, np.uint32, np.uint64],
        "float":   [np.float16, np.float32, np.float64],
        "complex": [np.complex64, np.complex128],
        "others":  [bool, object, bytes, str, np.void],
    }

# ── Compatibility Fix 2: PyTorch 2.6+ weights_only default ───────────────────
_original_torch_load = torch.load

def _patched_torch_load(*args, **kwargs):
    kwargs.setdefault("weights_only", False)
    return _original_torch_load(*args, **kwargs)

torch.load = _patched_torch_load

import importlib.util

from nemo.collections.asr.models.ctc_bpe_models import EncDecCTCModelBPE
from nemo.core.config import hydra_runner
from nemo.utils import logging
from nemo.utils.exp_manager import exp_manager

spec = importlib.util.spec_from_file_location(
    "trainer_utils", "NeMo/nemo/utils/trainer_utils.py"
)
trainer_utils = importlib.util.module_from_spec(spec)
spec.loader.exec_module(trainer_utils)
resolve_trainer_cfg = trainer_utils.resolve_trainer_cfg


@hydra_runner(config_path="../conf/citrinet/", config_name="config_bpe")
def main(cfg):
    logging.info(f"Hydra config:\\n{OmegaConf.to_yaml(cfg)}")

    trainer   = pl.Trainer(**resolve_trainer_cfg(cfg.trainer))
    exp_manager(trainer, cfg.get("exp_manager", None))
    asr_model = EncDecCTCModelBPE(cfg=cfg.model, trainer=trainer)

    asr_model.maybe_init_from_pretrained_checkpoint(cfg)

    trainer.fit(asr_model)

    if hasattr(cfg.model, "test_ds") and cfg.model.test_ds.manifest_filepath is not None:
        if asr_model.prepare_test(trainer):
            trainer.test(asr_model)


if __name__ == "__main__":
    main()
'''

with open(NEMO_SCRIPT_PATH, "w") as f:
    f.write(patched_script)

print(f"Patched script written to: {NEMO_SCRIPT_PATH}")

---
## Step 7 — Launch Fine-tuning



In [ ]:
!python /kaggle/working/NeMo/examples/asr/asr_ctc/speech_to_text_ctc_bpe.py \
    --config-path="/kaggle/working/" \
    --config-name="finetune_aug_config" \
    ++init_from_ptl_ckpt="/kaggle/input/datasets/jennypoudel/newari-data/Conformer-CTC-BPE--val_wer0.2177-epoch49.ckpt" \
    model.train_ds.manifest_filepath="/kaggle/input/augmented-data/augmentation/nemo_train.json" \
    model.validation_ds.manifest_filepath="/kaggle/working/mono_nemo_val.json" \
    model.test_ds.manifest_filepath="/kaggle/working/mono_nemo_test.json" \
    model.tokenizer.dir="/kaggle/input/datasets/jennypoudel/newari-data/" \
    model.tokenizer.type="bpe"